# How sure are you? — Group C: Has summer actually got hotter?

**The question:** Did the chance of a Tirana **June–August** day with maximum temperature **strictly above 35°C** change between **1961–1990** and **1991–2020**? Did the *mean* summer maximum or its *95th-percentile tail* shift?

**Rules:** no machine learning, no forecasts or predictive models, no scikit-learn; all probabilistic conclusions use ranges; 10,000 year-block bootstrap draws and 10,000 **year-label** permutations, fixed seeds; no network activity in this notebook. Raw data is read only from locally cached original responses. Exact observed counts are identified separately from uncertainty estimates (reported as `[n,n]` if a literal range is required). Public data credits appear below.

> **Pre-registration:** The actual prediction is in `config/SEALED_PREDICTION.md`; do not revise after source access. Empty placeholders are **not** results. We deliberately never touch the live sources in notebook cells.

## 0. Setup — offline by construction

Install `requirements.txt` **before** turning off network, then restart the kernel. The notebook imports only local project modules, file readers, plotting, pandas and NumPy. It cannot call the collector. All outputs are generated from `raw/` and saved under `processed/` and `outputs/`.

In [ ]:
from pathlib import Path
import json
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

candidates = (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent)
ROOT = next((p for p in candidates if (p / 'config' / 'team.json').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the how_sure_weather project tree.')
sys.path.insert(0, str(ROOT / 'src'))
from how_sure_weather.cache import read_manifest, load_cached
from how_sure_weather.data import (read_cached_daily, source_request_dates, data_coverage,
                                  summer_year_table, PERIODS)
from how_sure_weather.wiki import read_cached_wiki, load_period_confirmation
from how_sure_weather.statistics import (DRAWS, SEEDS, rate_bootstrap,
                                         permutation_null, mean_tail_bootstrap)
from how_sure_weather.sanity import compare_monthly_normals
from how_sure_weather.charts import permutation_histogram, change_comparison, period_rate_chart

(ROOT / 'processed').mkdir(exist_ok=True)
(ROOT / 'outputs').mkdir(exist_ok=True)
print('Offline-only analysis. Network requests from notebook: 0.')
print('Year-level random seeds:', SEEDS, '| draws per method:', DRAWS)

## 1. Access check (Monday 9 Nov) — verify the saved responses, never fetch

**Live action is a separate CLI step:** on Monday `python scripts/collect.py --stage access`. It must make exactly one request to each source (ERA5 probe and Tirana Wikipedia HTML). If either source fails or does not have a usable table: **stop and ask for help that day**. This notebook re-verifies the saved cache SHA-256 hashes. It does not contact either site.

In [ ]:
team = json.loads((ROOT / 'config' / 'team.json').read_text(encoding='utf-8'))
seal = (ROOT / 'config' / 'SEALED_PREDICTION.md').read_text(encoding='utf-8')
if 'SEALED, do not edit' not in seal:
    raise RuntimeError('Prediction not sealed; ask team to complete the pre-data step.')
if any(token in team['user_agent'] for token in ['[', ']', 'TEAM EMAIL', 'USER-AGENT']):
    raise RuntimeError('Real contact-based User-Agent not configured.')
access_report_path = ROOT / 'processed' / 'access_check_report.json'
if not access_report_path.exists():
    raise RuntimeError('Monday access check report absent. STOP: no live request from notebook.')
access_report = json.loads(access_report_path.read_text(encoding='utf-8'))
for key in ['open_meteo_probe', 'wikipedia_tirana']:
    blob = load_cached(ROOT, key)
    print(key, ': verified cached bytes; original response size [', len(blob), ',', len(blob), ']', sep='')
print('Reported access date (Tirana):', access_report['access_date_tirane'])
display(Markdown(seal))

## 2. Full collection (Tuesday 10 Nov) — load cached ERA5

**Only Person 1's CLI makes this request:** `python scripts/collect.py --stage full`. It requests the entire history **once**, with start `1940-01-01`, end set to *yesterday on the collection date*, and explicit `models=era5`, `timezone=Europe/Tirane`, `daily=temperature_2m_max,temperature_2m_min,precipitation_sum`. Any archive latency is reported as missing returned dates, never filled. The query URL, local access date, original JSON and SHA-256 stay in `raw/manifest.json`.

In [ ]:
full_blob = load_cached(ROOT, 'open_meteo_full')
daily = read_cached_daily(ROOT)
start_requested, end_requested = source_request_dates(ROOT)
coverage = data_coverage(daily, start_requested, end_requested)
(ROOT / 'processed' / 'coverage.json').write_text(json.dumps(coverage, indent=2) + '\n', encoding='utf-8')
daily.to_csv(ROOT / 'processed' / 'daily_weather.csv', index=False)
collection_report_path = ROOT / 'processed' / 'collection_report.json'
if not collection_report_path.exists():
    raise RuntimeError('Tuesday collection report absent. STOP and verify the raw response.')
print('Requested interval:', start_requested, 'to', end_requested)
for key in ['expected_calendar_days', 'returned_date_rows', 'missing_date_rows', 'null_max_among_returned', 'null_min_among_returned', 'null_precip_among_returned']:
    count = coverage[key]
    print(f'{key}: {count} [{count}, {count}] (exact count in cached archive/expected calendar)')
print('First missing dates, if any:', coverage['missing_date_list'][:8])
print('No missing date is filled. A missing max has nullable hot-day value (NA).')
display(daily.head(5))

## 3. Wikipedia scrape from Monday's cached Tirana page

Person 2 inspects the selected climate-table caption, source note, units, and row-specific normal years. Candidate years parsed from a caption are **unverified**. The script extracts mean daily maximum, mean daily minimum, mean temperature and precipitation **only if their rows are actually present**, and retains the original row labels. Another city is **not** used for Group C's main analysis. The Wikipedia table is monthly, so we never multiply its rows into the daily ERA5 dataset.

In [ ]:
wiki, wiki_meta = read_cached_wiki(ROOT)
wiki.to_csv(ROOT / 'processed' / 'wiki_normals.csv', index=False)
(ROOT / 'processed' / 'wiki_metadata.json').write_text(json.dumps(wiki_meta, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
print('Selected climate caption:', wiki_meta['chosen_caption'])
print('Source note:', wiki_meta['source_note'][:450] or '[not extractable from cached table]')
print('Caption normal-year candidates (NOT VERIFIED):', wiki_meta['caption_period_candidates'])
print('Selected row labels:', wiki_meta['row_labels'])
print('Verification status: UNVERIFIED until Person 2 confirms source/row/years.')
display(wiki.head(8))

## 4. Join plan, completeness, and data contract

**Grain:** daily ERA5 (`date`) → summer-year diagnostics (`year`) → 30-year periods (`period`); separately Wikipedia monthly normals (`month`) → same-years ERA5 monthly mean daily maxima (`month`), only after validating the Wikipedia normal years. Join on `month` **after** the matching-year ERA5 aggregation, not on each daily row.

**Contract:** `data_contract.csv` has every exported field's table name, dtype, unit, source and known gaps. 1991–2020 can be called **complete** only if each expected summer maximum is present. Keep 1961–1990 and 1991–2020 exact. Missing maximum values are not hot or non-hot — they are excluded from the observed denominator and counted separately.

In [ ]:
yearly = summer_year_table(daily)
yearly.to_csv(ROOT / 'processed' / 'summer_year.csv', index=False)
contract = pd.read_csv(ROOT / 'data_contract.csv', keep_default_na=False)
required = {'table', 'column', 'dtype', 'unit', 'source', 'known_gaps'}
assert set(contract.columns) == required
assert not (contract['known_gaps'].str.strip() == '').any()
period_coverage = yearly.groupby('period', sort=False)[['expected_days', 'returned_date_rows',
    'valid_max_days', 'missing_or_null_max_days', 'hot_days']].sum().reset_index()
display(period_coverage)
for label in [p[0] for p in PERIODS]:
    missing = int(period_coverage.loc[period_coverage.period.eq(label), 'missing_or_null_max_days'].iloc[0])
    print(f'{label}: archive summer maxima complete? {missing == 0}; missing/null count {missing} [{missing}, {missing}].')
print('Full data contract:', len(contract), 'column entries in data_contract.csv (schema metadata, not results).')
display(contract.head(12))

## 5. Explore *before* comparing periods

Explore only the two pre-registered 30-year comparison windows. Show exposure and year-level counts, not a fitted trend or forecast. A year's value is descriptive, not treated as an independent day-level draw. 2026 is excluded by construction. Missingness patterns and unusually small hot-day counts are highlighted rather than repaired.

In [ ]:
yearly['coverage_percent'] = 100 * yearly['valid_max_days'] / yearly['expected_days']
print('Year slots with any missing/null summer maximum:')
missing_years = yearly.loc[yearly['missing_or_null_max_days'] > 0, ['year', 'period', 'valid_max_days', 'missing_or_null_max_days']]
display(missing_years if not missing_years.empty else pd.DataFrame({'status': ['No missing summer maxima in either period']}))
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.bar(yearly['year'], yearly['coverage_percent'], width=.9)
ax.set(xlabel='Summer year', ylabel='Observed nonnull max coverage (%)',
       title='Coverage only: data completeness, not evidence of warming', ylim=(0, 105))
fig.tight_layout()
fig.savefig(ROOT / 'outputs' / 'summer_coverage.png', dpi=180)
plt.show()
plt.close(fig)
print('This plot is exact observed coverage; inferential ranges appear in Q1 and Q2.')

## 6. Question one — base rate and year-block bootstrap

**Probability method A: base rate.** Estimate \(P(\mathrm{max}>35^{\circ}C\mid\mathrm{June\!:\!August},\mathrm{period})\) as `observed hot days / observed nonmissing summer maxima`. Show numerator, denominator, expected-day completeness, rate and bootstrap 95% range for both periods.

**Probability method B: year-block bootstrap.** Within each pre-defined 30-year period, sample **whole years with replacement** 10,000 times (fixed seed), recompute numerator/denominator, then form the late-minus-early rate and its 95% percentile interval. If that difference interval includes zero, say **"we cannot tell"**, not "the difference is real." If there are few or no hot days, warn that empirical bootstrap intervals may under-represent unseen rare events.

In [ ]:
rates, rate_change = rate_bootstrap(yearly, draws=DRAWS, seed=SEEDS['hot_bootstrap'])
rates.to_csv(ROOT / 'processed' / 'period_rates.csv', index=False)
rate_change_row = {k: v for k, v in rate_change.items() if k != 'distribution'}
pd.DataFrame([rate_change_row]).to_csv(ROOT / 'processed' / 'rate_difference.csv', index=False)
for row in rates.itertuples(index=False):
    print(f'{row.period}: hot days {row.hot_days} [{row.hot_days},{row.hot_days}] exact observed; '
          f'valid summer maxima {row.valid_max_days} [{row.valid_max_days},{row.valid_max_days}] exact; '
          f'of {row.expected_days} [{row.expected_days},{row.expected_days}] expected; '
          f'rate {100*row.rate:.3f}% [95% year-bootstrap: {100*row.ci_lower:.3f}%, {100*row.ci_upper:.3f}%].')
print(f'Late minus early rate = {100*rate_change["observed_difference"]:.3f} percentage points '
      f'[95% year-bootstrap: {100*rate_change["ci_lower"]:.3f}, {100*rate_change["ci_upper"]:.3f} pp].')
rare_hot = any(rates['hot_days'] < 10)
if rare_hot:
    print('SPARSE HOT EVENTS WARNING: <10 observed hot days in at least one period. '
          'Keep the 35°C threshold; avoid exaggerated certainty. Zero-event bootstrap intervals cannot exclude unseen rare events.')
rate_evidence = bool(rate_change['ci_lower'] > 0 or rate_change['ci_upper'] < 0)
print('Q1 statement:', 'Interval excludes zero: measured shift supported (ERA5 only).' if rate_evidence
      else 'Interval includes zero: WE CANNOT TELL from this sample/method.')
fig = period_rate_chart(rates, ROOT / 'outputs' / 'period_rates.png')
plt.show()
plt.close(fig)

## 7. Question one — shuffle-year no-change chance baseline

**Probability method C: permutation / chance baseline.** Pool the sixty whole summer-year blocks. Shuffle **year labels**, assign 30 year slots to each period, compute the difference in hot-day base rates. Repeat 10,000 times with a fixed seed. The histogram is the no-period-difference **exchangeability baseline**, with the observed change marked. The central 95% shuffled range is a null range; it is **not** a forecast and not the confidence interval for a future day. Report the extreme fraction with a Wilson range describing Monte Carlo error only.

In [ ]:
shuffled = permutation_null(yearly, rate_change['observed_difference'], draws=DRAWS, seed=SEEDS['permutation'])
shuffle_row = {k: v for k, v in shuffled.items() if k != 'distribution'}
pd.DataFrame([shuffle_row]).to_csv(ROOT / 'processed' / 'permutation_summary.csv', index=False)
print(f'Central 95% of year-label shuffle differences: '
      f'[{100*shuffled["null_ci_lower"]:.3f}, {100*shuffled["null_ci_upper"]:.3f}] percentage points.')
print(f'Fraction of shuffled results at least as extreme as observed: {shuffled["extreme_fraction"]:.4f} '
      f'[Monte Carlo Wilson 95% range {shuffled["extreme_fraction_ci_lower"]:.4f}, '
      f'{shuffled["extreme_fraction_ci_upper"]:.4f}] (not a climate CI).')
fig = permutation_histogram(shuffled, ROOT / 'outputs' / 'permutation_histogram.png')
plt.show()
plt.close(fig)

## 8. Question two — did the mean or the hot tail move?

For each 30-year period, calculate (1) the pooled mean of nonmissing June–August daily maxima and (2) the pooled 95th percentile of those same days. Draw 10,000 bootstrap samples of **whole years within each period**; for each resample, calculate its mean and its 95th percentile. The plotted change is `1991–2020 – 1961–1990`, in degrees Celsius. **Call the change supported only if its bootstrap 95% interval excludes zero.** Do not claim a difference is real when its range crosses zero.

In [ ]:
changes, change_draws = mean_tail_bootstrap(daily, draws=DRAWS, seed=SEEDS['temp_bootstrap'])
changes.to_csv(ROOT / 'processed' / 'question_two_changes.csv', index=False)
for row in changes.itertuples(index=False):
    conclusion = 'interval excludes zero: evidence of a change in ERA5' if row.interval_excludes_zero else 'interval covers zero: WE CANNOT TELL'
    print(f'{row.metric}: shift {row.observed_change:+.3f} °C [95% year-bootstrap: '
          f'{row.ci_lower:+.3f}, {row.ci_upper:+.3f} °C]; {conclusion}.')
fig = change_comparison(changes, ROOT / 'outputs' / 'mean_vs_tail_change.png')
plt.show()
plt.close(fig)

## 9. Wikipedia normals sanity check — only after years are verified

The Wikipedia **Tirana** climate table's published monthly normal may refer to different years, a separate station, or separate rows with distinct normal periods. Person 2 must inspect the cached caption, row and source notes, and fill `config/wiki_period_confirmation.json` (copy from its example), confirming years **for the mean daily maximum row specifically** with an evidence quote.

Only after that can we aggregate ERA5 daily maxima to **the same years and calendar months**, then join the 12 ERA5 monthly averages to 12 Wikipedia published monthly normals. We bootstrap ERA5's mean by **year**; Wikipedia's published normal is a fixed published point with **no provided uncertainty interval**. Do *not* invent an uncertainty range for the Wikipedia source. Its difference from ERA5 is only a sanity check and must not be interpreted as a warming trend.

In [ ]:
confirmation = load_period_confirmation(ROOT, wiki_meta)
sanity = None
if confirmation is None:
    print('SANITY CHECK WITHHELD: Wikipedia mean-maximum normal years are not independently verified.')
    print('Action: confirm the relevant caption/row/source and create config/wiki_period_confirmation.json.')
else:
    wiki_first, wiki_last = confirmation
    sanity = compare_monthly_normals(daily, wiki, wiki_first, wiki_last,
        draws=DRAWS, seed=SEEDS['sanity_bootstrap'])
    sanity.to_csv(ROOT / 'processed' / 'sanity_monthly.csv', index=False)
    print(f'Comparable Wikipedia normal period (manually verified): {wiki_first}–{wiki_last}.')
    print('Difference ranges are conditional on the Wikipedia point values (whose uncertainty is UNSPECIFIED).')
    display(sanity[['month', 'wiki_mean_max_c', 'era5_mean_max_c',
                    'difference_c', 'difference_ci_lower_c', 'difference_ci_upper_c',
                    'missing_or_null_max_days']])
    fig, ax = plt.subplots(figsize=(9, 4.5))
    # Draw uncertainty segments separately; a percentile interval might not surround the point.
    ax.vlines(sanity['month'], sanity['difference_ci_lower_c'],
              sanity['difference_ci_upper_c'], linewidth=2)
    ax.scatter(sanity['month'], sanity['difference_c'], s=35)
    ax.axhline(0, linestyle='--')
    ax.set(xticks=list(range(1, 13)), xlabel='Month', ylabel='ERA5 − Wikipedia normal (°C)',
           title=f'Same-year monthly sanity check ({wiki_first}–{wiki_last}); ERA5-only 95% ranges')
    fig.tight_layout()
    fig.savefig(ROOT / 'outputs' / 'wikipedia_sanity_check.png', dpi=180)
    plt.show()
    plt.close(fig)

## 10. Decisions (pre-registered and data-dependent)

These statements must accompany the final story. The predetermined parts may not be revised after inspecting data.

1. **Periods:** 1961–1990 vs 1991–2020, each inclusive. Confirm whether 1991–2020 has all 30 × 92 expected daily summer maximum values. Show missing/null counts.
2. **Hot:** daily maximum **strictly `>35°C`**. No retroactive threshold tuning.
3. **Summer:** June, July, August only. Exclude the current year 2026 even though it is outside the pre-registered comparison windows.
4. **Missing:** count unreturned dates separately from null maxima. Never impute; observed hot-day denominator counts nonmissing maxima only.
5. **Grid limits:** Open-Meteo ERA5 is a roughly 25 km reconstruction, not a station measurement; its extreme highs can be smoothed.
6. **Sparse-event rule:** if either period has fewer than ten observed hot days, label rates/tails sparse. Keep honest intervals. Zero observed hot days do not prove the physical risk is zero.
7. **Wikipedia:** only compare same years after Person 2 verifies the specific mean-daily-maximum normal period; unknown period means sanity check withheld.
8. **Evidence:** interval containing zero → **"we cannot tell"**; excluding zero → evidence *within this data and resampling procedure*, not a causal attribution.

The cell below writes `processed/decisions.md` and `outputs/talk_numbers.md` using **computed values**, never placeholders masquerading as data.

In [ ]:
rate_a = rates.iloc[0]
rate_b = rates.iloc[1]
def statement(low, high):
    return 'range EXCLUDES zero (ERA5 measured shift supported)' if low > 0 or high < 0 else 'range COVERS zero (WE CANNOT TELL)'

decisions_text = f"""# Decisions from the cached archive

- Fixed years: 1961–1990 and 1991–2020 (30 summers each); current year excluded.
- Fixed hot-day definition: strictly >35°C in June, July, August.
- 1991–2020 archive summer completeness: **{bool(rate_b['complete'])}**; missing/null summer maxima: {int(rate_b['missing_or_null_max_days'])} [{int(rate_b['missing_or_null_max_days'])},{int(rate_b['missing_or_null_max_days'])}] exact archive count.
- 1961–1990 missing/null summer maxima: {int(rate_a['missing_or_null_max_days'])} [{int(rate_a['missing_or_null_max_days'])},{int(rate_a['missing_or_null_max_days'])}] exact archive count.
- The archive had {int(coverage['missing_date_rows'])} [{int(coverage['missing_date_rows'])},{int(coverage['missing_date_rows'])}] unreturned dates within the full requested range; no imputation.
- ERA5 model: ~25 km gridded reanalysis, not thermometer station; high extremes may be smoothed.
- Sparse hot events (<10 in either period): **{rare_hot}**; fixed 35°C threshold retained and sparse-event uncertainty disclosed.
- Q1: {statement(rate_change['ci_lower'], rate_change['ci_upper'])}.
- Wikipedia normal years: **{str(confirmation) if confirmation is not None else 'NOT VERIFIED — sanity check withheld'}**.
- Zero-width count intervals are literal bounds of exact *cached* counts, not uncertainty intervals for the climate.
"""
(ROOT / 'processed' / 'decisions.md').write_text(decisions_text, encoding='utf-8')
talk_text = f"""# Actual computed talk numbers — use with data and intervals

**Source dates:** cached archive request {start_requested} to {end_requested}. Wikipedia access date {access_report['access_date_tirane']}. Wikipedia normal years: {str(confirmation) if confirmation else '[UNVERIFIED — sanity check withheld]'}.

**Q1 (observed cache counts and 95% year-bootstrap rate ranges):**
- 1961–1990: {int(rate_a['hot_days'])} [{int(rate_a['hot_days'])},{int(rate_a['hot_days'])}] hot / {int(rate_a['valid_max_days'])} [{int(rate_a['valid_max_days'])},{int(rate_a['valid_max_days'])}] valid observed days, of {int(rate_a['expected_days'])} expected; rate {100*rate_a['rate']:.3f}% [{100*rate_a['ci_lower']:.3f}%, {100*rate_a['ci_upper']:.3f}%].
- 1991–2020: {int(rate_b['hot_days'])} [{int(rate_b['hot_days'])},{int(rate_b['hot_days'])}] hot / {int(rate_b['valid_max_days'])} [{int(rate_b['valid_max_days'])},{int(rate_b['valid_max_days'])}] valid observed days, of {int(rate_b['expected_days'])} expected; rate {100*rate_b['rate']:.3f}% [{100*rate_b['ci_lower']:.3f}%, {100*rate_b['ci_upper']:.3f}%].
- Difference late−early: {100*rate_change['observed_difference']:+.3f} percentage points [{100*rate_change['ci_lower']:+.3f}, {100*rate_change['ci_upper']:+.3f}] 95% bootstrap; {statement(rate_change['ci_lower'], rate_change['ci_upper'])}.
- Permutation central 95% shuffle differences: [{100*shuffled['null_ci_lower']:+.3f}, {100*shuffled['null_ci_upper']:+.3f}] percentage points. Extreme shuffle fraction: {shuffled['extreme_fraction']:.4f} [Monte Carlo error {shuffled['extreme_fraction_ci_lower']:.4f}, {shuffled['extreme_fraction_ci_upper']:.4f}].

**Q2 (95% year-bootstrap changes in °C):**
"""
for r in changes.itertuples(index=False):
    talk_text += (f'- {r.metric}: {r.observed_change:+.3f}°C [{r.ci_lower:+.3f}, {r.ci_upper:+.3f}] °C; '
                  f'{statement(r.ci_lower,r.ci_upper)}.\n')
talk_text += (f'\n**Missing:** 1961–1990 {int(rate_a["missing_or_null_max_days"])} exact missing/null summer maxima; '
              f'1991–2020 {int(rate_b["missing_or_null_max_days"])} exact. '
              f'Sparse-event caution: {rare_hot}.\n'
              '\n**Limits:** ERA5 grid not station; potential smoothing, missing days not filled, year exchangeability simplification, Wikipedia normal-period/measurement mismatch.\n')
(ROOT / 'outputs' / 'talk_numbers.md').write_text(talk_text, encoding='utf-8')
summary = {
    'rate_difference': rate_change_row,
    'permutation': shuffle_row,
    'question_two_changes': json.loads(changes.to_json(orient='records')),
    'wikipedia_confirmed_period': list(confirmation) if confirmation else None,
    'coverage': {k: v for k, v in coverage.items() if k != 'missing_date_list'},
    'rare_hot_day_warning': rare_hot,
}
(ROOT / 'processed' / 'analysis_summary.json').write_text(json.dumps(summary, indent=2) + '\n', encoding='utf-8')
display(Markdown(decisions_text))
print('Slide numbers written to outputs/talk_numbers.md — copy only after checking the ranges.')

## 11. Limits, attribution, and reproducibility

- **Small hot-day counts:** the 35°C tail may be sparse in coarse ERA5 grid data. Wide ranges are valid findings. Empirical bootstrap cannot infer previously unseen rare events from all-zero samples.
- **Spatial measurement:** ERA5 is reconstructed/reanalysed on a coarse grid, not station readings; high daytime extremes can be smoothed. No causal claim or forecast is made.
- **Coverage:** API gaps and null daily maxima are counted, not filled. Periods with differential missingness are not directly equivalent samples of calendar days.
- **Dependence:** resample and shuffle **whole summer-year blocks**. This respects within-year clustering but not all multiyear effects or changing measurement conditions.
- **Permutation baseline:** year-label exchangeability is a statistical hypothetical, not a simulated climate.
- **Wikipedia:** the Tirana table reports published monthly normals and may use different station years. Only compare the exact matching period if verified, and never interpret a Wikipedia–ERA5 discrepancy as proof that summers warmed.
- **Language rule:** intervals crossing zero → **"we cannot tell"**. Evidence and uncertainty always travel together.

**Data credit:** Open-Meteo historical ERA5 archive at Tirana coordinates; ERA5 from ECMWF/Copernicus, attributed in accordance with CC BY 4.0 stated in the brief. Wikipedia contributors, *Tirana* climate box and cited underlying weather-data provider, exact cached page and access date in `raw/manifest.json`. Original raw responses are kept unchanged and must be committed. No data has been downloaded by this notebook.

**Sunday offline proof:** Disconnect Wi-Fi; restart kernel; **Run All**. Alternatively, with the network physically disabled, run `OFFLINE=1 python scripts/offline_check.py`. Confirm every cell succeeds, all hashes match, contract fields match exports, sanity years are verified or explicitly withheld, and no uncached request or placeholder result is present.